In [1]:
import os
import sys
import json
import numpy as np
import cupy as cp
import laspy
import pyproj
from pathlib import Path
import simplejson as sjson

module_path = os.path.abspath(os.path.join('..'))
if module_path not in sys.path:
    sys.path.append(module_path)

from workflow.utils.logger import logger
from workflow.pointcloud import PointCloud as PC
from workflow.processor.converter.point_cloud_rasterization import point_cloud_rasterization
from workflow.processor.cutting.cut_point_cloud_by_polygon import cut_point_cloud_by_polygon

Jupyter environment detected. Enabling Open3D WebVisualizer.
[Open3D INFO] WebRTC GUI backend enabled.
[Open3D INFO] WebRTCWindowSystem: HTTP handshake server disabled.


In [2]:
def order_polygon_points(xy: np.ndarray) -> np.ndarray:
    """质心+极角排序"""
    c = xy.mean(axis=0)
    ang = np.arctan2(xy[:, 1] - c[1], xy[:, 0] - c[0])  # 返回点 (dx, dy) 的极角（范围 -π 到 π）
    return xy[np.argsort(ang)]

In [3]:
def convert2geo_coord(parsed_points, crs, min_x, max_x, min_y, max_y, need_order: bool=False):
    """
    将LabelStudio中的百分比坐标转换为实际的地理坐标。
    
    参数：
    - parsed_points: 从LabelStudio导出的json读出的单个管辖区百分比坐标
    - min_x, max_x: 点云投影的实际地理坐标范围（x轴）。
    - min_y, max_y: 点云投影的实际地理坐标范围（y轴）。
    
    返回：
    - 转换后的cupy地理坐标点
    """
    assert crs == 'EPSG:4547', f'坐标系为{crs}，XY轴朝向需要进一步确定'

    if need_order:  # 多边形坐标是否排好序，默认情况下都是排好序的
        parsed_points =  order_polygon_points(parsed_points)

    # 统一转为 python float，避免 numpy/cupy 标量混入 JSON
    min_x = float(min_x); max_x = float(max_x)
    min_y = float(min_y); max_y = float(max_y)
    
    geo_coordinates = []

    for point in parsed_points:
        # 百分比坐标，需除以100；且labelstudio中的图像坐标轴原点在左上角，y轴要取反
        geo_x = min_x + (max_x - min_x) * point[0] / 100
        geo_y = max_y - (max_y - min_y) * point[1] / 100
        
        geo_coordinates.append((geo_x, geo_y))

    return geo_coordinates

In [4]:
def to_bool_mask(mask, n_points):
    """
        将输入统一转为 cupy.ndarray 类型的布尔掩码。
        支持:
        - cupy.ndarray (bool 或 int)
        - numpy.ndarray (bool 或 int)
        - list/tuple (bool 或 int)
    """

    mask = cp.asarray(mask)

    if mask.dtype == cp.bool_:
        if mask.shape[0] != n_points:
            raise ValueError(f"Boolean mask length {mask.shape[0]} ≠ point count {n_points}")
        return mask

    if np.issubdtype(mask.dtype, np.integer):
        # 索引 -> bool 掩码
        bool_mask = np.zeros(n_points, dtype=bool)
        # 过滤越界索引（如果有）
        valid = (mask >= 0) & (mask < n_points)
        bool_mask[mask[valid]] = True
        return bool_mask

    raise TypeError("mask 必须是布尔数组或整型索引数组/列表。")

In [5]:
def save_masked_las(mask, n, input_path, output_path):
    """
    读取 input_path（.las/.laz），按 mask 过滤点，保留元信息，写到 output_path。
    - 保留：header（含 scales/offsets/point_format/CRS/VLR）、EVLR、所有点属性（含 extra dims）
    - mask 可为布尔数组或整型索引（一维）
    """
    input_path = Path(input_path)
    output_path = Path(output_path)
    output_path.mkdir(parents=False, exist_ok=True)

    las = laspy.read(str(input_path))

    bool_mask = to_bool_mask(mask, n)

    # 过滤点：las.points 是结构化数组，切片后保留所有维度（包括额外维度）
    filtered_points = las.points[bool_mask]

    # 复制 header（包含 scales/offsets/point_format/VLR/CRS 等）
    header = las.header.copy()

    # 用复制的 header 创建新 LasData，并设置点数据
    new_las = laspy.LasData(header)
    new_las.points = filtered_points  # 会自动更新点计数与 per-return 统计

    # 复制 EVLR（注意：header.copy() 不包含 EVLR，需要单独带上）
    if hasattr(las, "evlrs") and las.evlrs is not None:
        new_las.evlrs.extend(las.evlrs)

    new_las.write(str(output_path))

    return {
        "seg_points": int(bool_mask.sum()),
        "point_format": str(las.header.point_format),
        "has_crs": new_las.header.parse_crs(prefer_wkt=False) is not None
    }

In [6]:
# 给边坡的所有标定管辖区做坐标转换并保存

project_name = 'DJI_202602141143_001_20260214学校操场水平-重复-无植被-无垫高'
las_path = Path('/home/gary/CloudPointProcessing/20260214学校操场实验') / project_name / 'raw' / 'las' / 'cloud_merged.las'
json_file = f'/home/gary/point-cloud-3d/output/管辖区坐标/20260214学校操场实验/植被选区_百分比坐标.json'

pc = PC()
pc.load_from_las(las_path)
pc.transform_to(pyproj.CRS.from_epsg(4547))

p = Path(json_file)
if '地理坐标' in p.name:
    logger.warning(f"文件{json_file}已包含地理坐标，不需转换")
else:
    text = p.read_text(encoding='utf-8')
    json_data = json.loads(text)  # 从字符串中加载json中的浮点数坐标

    # with open(json_file, 'r', encoding='utf-8') as f:  # 这种方式可能有损精度，原因待定
    #         json_data = json.load(f)

    for i, res in enumerate(json_data['annotations'][0]['result']):
        parsed_points = res['value']['points']
        polygon_xys = convert2geo_coord(parsed_points, pc.crs, cp.min(pc.x), cp.max(pc.x), cp.min(pc.y), cp.max(pc.y), need_order=False)
        
        # 只修改json_data中的坐标信息，其他信息不变
        json_data['annotations'][0]['result'][i]['value']['points'] = polygon_xys
    
    with open(json_file.replace('百分比坐标', '地理坐标'), 'w') as f:
        sjson.dump(json_data, f, use_decimal=True)

/home/gary/anaconda3/envs/PC/lib/python3.12/site-packages/cupy/_creation/from_data.py:88: PerformanceWarning: Using synchronous transfer as pinned memory (530292464 bytes) could not be allocated. This generally occurs because of insufficient host memory. The original error was: cudaErrorMemoryAllocation: out of memory
  return _core.array(a, dtype, False, order, blocking=blocking)


In [7]:
# # 直接使用投影坐标，按边坡单独分割保存

# json_file = json_file.replace('百分比坐标', '地理坐标')

# p = Path(json_file)
# text = p.read_text(encoding='utf-8')
# json_data = json.loads(text)  # 从字符串中加载json中的浮点数坐标

# for i, res in enumerate(json_data['annotations'][0]['result']):
#     parsed_points = res['value']['points']
#     parsed_points = cp.array(parsed_points, dtype=float)

#     seg_pc, mask = cut_point_cloud_by_polygon(pc, parsed_points, backend='cupy', include_boundary=True)

#     # check_point_cloud(seg_pc)
    
#     # parts = project_name.split("_")
#     # date = parts[0]
#     # direction = parts[2]
#     # road = parts[3]

#     point_cloud_rasterization(
#         seg_pc,
#         sampling_mode='max_z',
#         sampling_rate=2000,
#         enable_fill=False, 
#         enable_plt=False, 
#         fill_algorithm='nearest', 
#         # save_path=f'/home/point-cloud-3d/point-cloud-3d/output/{direction}_{road}_{date}_管辖区{i}.png'
#         save_path=f'/home/point-cloud-3d/point-cloud-3d/output/{project_name}.png'
#         )

#     # TODO: 速度太慢了，暂时不转las，临时分割即可
#     # seg_points, point_format, has_crs = save_masked_las(seg_pc, len(pc), las_path, las_path.parent.parent / 'seg_pc_las')
#     # logger.info(f"seg_points = {seg_points}, point_format = {point_format}, has_crs= {has_crs}")